In [2]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

In [3]:
import wandb
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow import keras as k
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

/Users/kavin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/kavin/.netrc.
wandb: Currently logged in as: kav13 (kav13-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
class LogLRCallback(k.callbacks.Callback):
    """Log the optimizer learning rate at the end of every epoch."""

    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val}, step=self.model.optimizer.iterations.numpy())

In [6]:
class LogSamplesCallback(k.callbacks.Callback):
    """Log a table of sample predictions and images every epoch."""

    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(
            columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"]
        )
        for i in range(len(self.x)):
            img = (self.x[i].squeeze() * (255.0 / 16.0)).astype(np.uint8)
            table.add_data(
                wandb.Image(img),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch + 1}": table})

In [7]:
class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a confusion matrix from the validation set each epoch."""

    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})

In [8]:
class DigitsTrainer:
    """Train a CNN on sklearn Digits with full WandB experiment tracking."""

    LABELS = [str(i) for i in range(10)]

    def __init__(
        self,
        project_name="Lab2-Sklearn-Digits-CNN",
        run_name="digits_cnn_v1",
    ):
        self.cfg = dict(
            dropout=0.25,
            conv1_filters=32,
            conv2_filters=64,
            dense_units=128,
            learn_rate=0.01,
            momentum=0.9,
            epochs=30,
            batch_size=32,
            test_size=0.2,
            random_state=42,
        )
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=self.cfg,
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self._prepare_data()

    # ------------------------------------------------------------------
    # Data
    # ------------------------------------------------------------------
    def _prepare_data(self):
        digits = load_digits()
        X = digits.images.astype("float32") / 16.0
        y = digits.target

        X = X[..., np.newaxis]  # (N, 8, 8) -> (N, 8, 8, 1)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y,
        )

        self.X_train = X_train
        self.X_test = X_test
        self.y_train = to_categorical(y_train)
        self.y_test = to_categorical(y_test)
        self.num_classes = self.y_test.shape[1]

        wandb.log({
            "data/train_samples": len(X_train),
            "data/test_samples": len(X_test),
            "data/num_classes": self.num_classes,
            "data/image_shape": str(X_train.shape[1:]),
        })

    # ------------------------------------------------------------------
    # Model — lightweight CNN suited for 8x8 grayscale
    # ------------------------------------------------------------------
    def _build_model(self):
        inputs = k.Input(shape=(8, 8, 1))

        # Block 1
        x = k.layers.Conv2D(
            self.config.conv1_filters, (3, 3), activation="relu", padding="same"
        )(inputs)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        # Block 2
        x = k.layers.Conv2D(
            self.config.conv2_filters, (3, 3), activation="relu", padding="same"
        )(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        # Head
        x = k.layers.Flatten()(x)
        x = k.layers.Dense(self.config.dense_units, activation="relu")(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)

        model = k.Model(inputs, outputs)

        opt = k.optimizers.SGD(
            learning_rate=self.config.learn_rate,
            momentum=self.config.momentum,
            nesterov=True,
        )
        model.compile(
            loss="categorical_crossentropy",
            optimizer=opt,
            metrics=["accuracy"],
        )
        return model

    # ------------------------------------------------------------------
    # Artifact logging
    # ------------------------------------------------------------------
    def _log_model_artifact(self, model):
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)

        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/model.h5"
        model.save(model_path)

        art = wandb.Artifact("digits_cnn_model", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    # ------------------------------------------------------------------
    # Train
    # ------------------------------------------------------------------
    def train(self):
        model = self._build_model()

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint(
                "checkpoints/model-{epoch:02d}.h5", save_weights_only=False
            ),
            LogLRCallback(),
            LogSamplesCallback(
                self.X_test, self.y_test, self.LABELS, max_rows=32
            ),
            ConfusionMatrixCallback(
                self.X_test, self.y_test, self.LABELS
            ),
        ]

        model.fit(
            self.X_train,
            self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final/loss": loss, "final/accuracy": acc})

        self._log_model_artifact(model)
        self.run.finish()
        print(f"\nDone — final val accuracy: {acc:.4f}")

In [9]:
trainer = DigitsTrainer()
trainer.train()

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Epoch 1/30
43/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.1087 - loss: 2.3028   

45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.1102 - loss: 2.3021 - val_accuracy: 0.3861 - val_loss: 2.2434
Epoch 2/30
34/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1916 - loss: 2.2467 

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.1975 - loss: 2.2385 - val_accuracy: 0.5556 - val_loss: 2.0347
Epoch 3/30
35/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3510 - loss: 2.0302 

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3584 - loss: 1.9964 - val_accuracy: 0.7333 - val_loss: 1.2595
Epoch 4/30
32/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5498 - loss: 1.4009 

...

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9806 - loss: 0.0658 - val_accuracy: 0.9861 - val_loss: 0.0415
Epoch 29/30
39/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9743 - loss: 0.0648 

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9747 - loss: 0.0655 - val_accuracy: 0.9861 - val_loss: 0.0460
Epoch 30/30
38/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9782 - loss: 0.0661 

45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.9784 - loss: 0.0659 - val_accuracy: 0.9833 - val_loss: 0.0448


batch/accuracy,▁▁▁▂▃▅▆▇▇▇█▇▇▇██████████████████████████
batch/batch_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,███▇▇▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
data/num_classes,▁
data/test_samples,▁
data/train_samples,▁
epoch/accuracy,▁▂▃▅▆▇▇▇▇█████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+6,...



Done — final val accuracy: 0.9833
